# Benchmark Temporal Splitting

This notebook implements the benchmark using **Temporal (Chronological)** splitting:
- **Train**: 0% - 70% of timeline
- **Val**: 70% - 85% of timeline  
- **Test**: 85% - 100% of timeline

This prevents data leakage and tests true extrapolation capability.

In [1]:
%load_ext autoreload
%autoreload 2

import os
import gc
import pandas as pd
import numpy as np
import torch
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import DataLoader
import xgboost as xgb
from sklearn.multioutput import MultiOutputRegressor

# Import from src
from src.data_processing import generate_cyclical_features, scale_data_selectively
from src.splitting import get_temporal_splits
from src.models import TinyTGT, GlobalFitLocalApplySARIMA
from src.evaluation import RollingDataset, prepare_xgb_data, run_evaluation, print_metrics, get_scaling_factors

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cuda


In [2]:
CONFIG = {
    # Date_Range that will be used to generate the temporal features 
    "START_DATE": "2024-01-01",
    "FREQUENCY": "H",                 
    
    # Dataset
    "DATA_PATH": "data_out/no_pertubations/case14_ieee/raw", 
    
    # Features
    "USE_ONLY_LOAD": True,              # Only use active load P
    "BINARY_ADJACENCY": True,           # True = Graph is 0/1 (no admittance)

    #TGT
    "BATCH_SIZE": 32,                  # Batch Size
    "EPOCHS": 15,                      # Number of Epochs
    
    # Forecasting  (Triebe et al. Logic)
    "INPUT_WINDOW": 168,                # Lookback: 7 Tage
    "FORECAST_HORIZON": 33,             # Predict: Rest of Day (9h) + Next Day (24h)
    "TARGET_DAY_START_IDX": 9,          # Target Day" (00:00 tmrw)
    "EVAL_HOUR": 14,                    # forecasting is done always at 2pm    
    # Baselines
    "SNAIVE_LAG": 48                    # Benchmark: value 48h ago
}

In [3]:
# Load Bus and Branch DataFrames
# Adjusted paths relative to the notebook location
bus_df = pd.read_parquet('../data_out/no_pertubations/case14_ieee/raw/bus_data.parquet')
branch_df = pd.read_parquet('../data_out/no_pertubations/case14_ieee/raw/branch_data.parquet')

# Build Edge Index from Branch Data
structure = branch_df[branch_df['load_scenario_idx'] == 0].sort_values("idx")
edge_index = list(structure[['from_bus', 'to_bus']].itertuples(index=False, name=None))

# print dims
m = len(edge_index)
n=bus_df['bus'].nunique()
print(f"Topology Loaded: n={n} Nodes, m={m} Edges")

# X: Input Loads (Active Power 'Pd')
loads_pivot = bus_df.pivot(index='load_scenario_idx', columns='bus', values='Pd').fillna(0)
# Y: Target Flows (Active Power Flow 'pf')
flows_pivot = branch_df.pivot(index='load_scenario_idx', columns='idx', values='pf').fillna(0)

# Convert to Numpy Arrays
loads_matrix = loads_pivot.values.astype(np.float32)
flows_matrix = flows_pivot.values.astype(np.float32)
print(f"shapes: Loads: {loads_matrix.shape}, Flows: {flows_matrix.shape}")

# Adjacency mask (neighbors + self)
A = np.zeros((n,n), dtype=bool)
for (i,j) in edge_index:
    A[i,j]=True; A[j,i]=True
for i in range(n): A[i,i]=True
A_mask = torch.from_numpy(A).to(device)

Topology Loaded: n=14 Nodes, m=20 Edges
shapes: Loads: (8760, 14), Flows: (8760, 20)


In [4]:
# --- Prepare Input Tensor X --- 
# 1. gen Time Features 
T = loads_matrix.shape[0]
time_features_global = generate_cyclical_features(T, CONFIG["START_DATE"], CONFIG["FREQUENCY"]) # [T, 6]
print(f"Time Features generated. Shape: {time_features_global.shape}")

# 2. gen input tensor X
load_tensor = loads_matrix[:, :, np.newaxis]  # [T, N] -> [T, N, 1]
time_tensor_expanded = np.broadcast_to(
    time_features_global[:, np.newaxis, :], 
    (time_features_global.shape[0], loads_matrix.shape[1], time_features_global.shape[1])
) # [T, 6] -> [T, N, 6]  (*read-only view)
X = np.concatenate([load_tensor, time_tensor_expanded], axis=2) # [T, N, 1] + [T, N, 6] -> [T, N, 7]
print(f"Final Input Tensor Shape: {X.shape} (Time, Buses, Features)")

# --- Split Data ---
train_idx, val_idx, test_idx = get_temporal_splits(T)
print(f"Train Hours: {len(train_idx)}")
print(f"Val Hours:   {len(val_idx)}")
print(f"Test Hours:  {len(test_idx)}")

# --- 3. Scaling (Only P not the sin/cos transformed time features) ---
scaled_tensor, scaler = scale_data_selectively(X, train_idx)
print(f"Scaled Mean (Load): {scaled_tensor[:,:,0].mean():.4f}")
print(f"Scaled Mean (Hour_Sin): {scaled_tensor[:,:,1].mean():.4f} (Should be near 0 but unscaled)")

Time Features generated. Shape: (8760, 6)
Final Input Tensor Shape: (8760, 14, 7) (Time, Buses, Features)
Train Hours: 6132
Val Hours:   1314
Test Hours:  1314
Scaled Mean (Load): -0.0194
Scaled Mean (Hour_Sin): 0.0000 (Should be near 0 but unscaled)


f:\studium\Thesis_Repo\phase1_tgt_model\src\data_processing.py:10: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  dates = pd.date_range(start=start_date, periods=T, freq=frequency)


In [5]:
# --- Create Datasets & Loaders ---
train_dataset = RollingDataset(scaled_tensor, train_idx, CONFIG["INPUT_WINDOW"], CONFIG["FORECAST_HORIZON"])
val_dataset   = RollingDataset(scaled_tensor, val_idx,   CONFIG["INPUT_WINDOW"], CONFIG["FORECAST_HORIZON"])

train_loader = DataLoader(train_dataset, batch_size=CONFIG["BATCH_SIZE"], shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=CONFIG["BATCH_SIZE"], shuffle=False)

# Test One Batch
x_batch, y_batch = next(iter(train_loader))
print(f"Batch Shapes:")
print(f"X (Input):  {x_batch.shape}  -> [Batch, 168, 14, 7]")
print(f"Y (Target): {y_batch.shape}  -> [Batch, 33, 14, 7]")

Batch Shapes:
X (Input):  torch.Size([32, 168, 14, 7])  -> [Batch, 168, 14, 7]
Y (Target): torch.Size([32, 33, 14, 7])  -> [Batch, 33, 14, 7]


In [6]:
# --- XGBoost Training ---
X_train_xgb, Y_train_xgb = prepare_xgb_data(
    scaled_tensor, 
    train_idx, 
    CONFIG["INPUT_WINDOW"], 
    CONFIG["FORECAST_HORIZON"],
    step_size=1
)
print(f"XGB Train Shape: {X_train_xgb.shape}")

xgb_estimator = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=4,
    objective='reg:squarederror',
    device='cuda',       
    tree_method='hist',
    n_jobs=4
)

xgb_model = MultiOutputRegressor(xgb_estimator, n_jobs=1)

print("Training Global XGBoost...")
xgb_model.fit(X_train_xgb, Y_train_xgb)
print("XGBoost Training Complete.")

Allocating RAM for 83496 samples...


Building XGB Dataset: 100%|██████████| 5964/5964 [00:00<00:00, 32237.68it/s]


XGB Train Shape: (83496, 175)
Training Global XGBoost...
XGBoost Training Complete.


In [7]:
# --- TinyTGT Training ---
torch.cuda.empty_cache() 

tgt_model = TinyTGT(n_nodes=14, d_model=64, n_heads=4, in_feat=7, out_steps=33).to(device)
opt = optim.Adam(tgt_model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

for ep in range(1, CONFIG["EPOCHS"]+1):
    tgt_model.train()
    train_loss, batch_count = 0.0, 0
    
    for xb, yb in train_loader:
        xb = xb.float().to(device) 
        yb = yb.float().to(device) 
        
        opt.zero_grad()
        yhat = tgt_model(xb, A_mask) 
        loss = loss_fn(yhat, yb[..., 0])
        loss.backward()
        opt.step()
        
        train_loss += loss.item()
        batch_count += 1
    
    # Validation
    tgt_model.eval()
    val_loss, val_batches = 0.0, 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.float().to(device)
            y_true = yb[..., 0].float().to(device)
            yhat = tgt_model(xb, A_mask)
            val_loss += loss_fn(yhat, y_true).item()
            val_batches += 1
            
    print(f"Epoch {ep}: Train MSE={train_loss/batch_count:.6f} | Val MSE={val_loss/val_batches:.6f}")

Epoch 1: Train MSE=0.126538 | Val MSE=0.023912
Epoch 2: Train MSE=0.026151 | Val MSE=0.018499
Epoch 3: Train MSE=0.018878 | Val MSE=0.013488
Epoch 4: Train MSE=0.014208 | Val MSE=0.011031
Epoch 5: Train MSE=0.010770 | Val MSE=0.009718
Epoch 6: Train MSE=0.009354 | Val MSE=0.008511
Epoch 7: Train MSE=0.007958 | Val MSE=0.008027
Epoch 8: Train MSE=0.007242 | Val MSE=0.007429
Epoch 9: Train MSE=0.006815 | Val MSE=0.006671
Epoch 10: Train MSE=0.006329 | Val MSE=0.007279
Epoch 11: Train MSE=0.006049 | Val MSE=0.005759
Epoch 12: Train MSE=0.005927 | Val MSE=0.006070
Epoch 13: Train MSE=0.005621 | Val MSE=0.006418
Epoch 14: Train MSE=0.005443 | Val MSE=0.006382
Epoch 15: Train MSE=0.005331 | Val MSE=0.005969


In [ ]:
# --- SARIMA Global Fit ---
# For temporal split: use ALL training data (contiguous, no gaps)
sarima_model = GlobalFitLocalApplySARIMA(order=(2, 0, 0), seasonal_order=(1, 0, 1, 24))
sarima_model.fit(scaled_tensor, train_idx, None)

NameError: name 'GlobalFitLocalApplySARIMA' is not defined

In [ ]:
# --- Evaluation ---
denom_mae, denom_mse = get_scaling_factors(scaled_tensor, train_idx, scaler, m=24)
print(f"Scaling Baseline (Train): MAE={denom_mae:.4f}, MSE={denom_mse:.4f}")

results = run_evaluation(scaled_tensor, test_idx, xgb_model, tgt_model, sarima_model, A_mask, scaler, CONFIG, device=device)


Scaling Baseline (Train): MAE=0.6622, MSE=2.1869
Evaluating 54 days...


100%|██████████| 54/54 [00:22<00:00,  2.36it/s]


=== FINAL BENCHMARK RESULTS ===
Model      | RMSE       | MAE        | MAPE       | MASE       | MSSE      
---------------------------------------------------------------------------
SNAIVE     | 2.2148     | 1.0126     | 8.85%     | 1.5293     | 2.2429
XGB        | 1.6793     | 0.7821     | 7828327.09%     | 1.1811     | 1.2895
SARIMA     | 1.8523     | 0.8924     | 8.59%     | 1.3477     | 1.5688
TinyTGT    | 1.8341     | 0.9797     | 28271379.35%     | 1.4795     | 1.5383
---------------------------------------------------------------------------


In [13]:
print_metrics(results, denom_mae, denom_mse)


=== FINAL BENCHMARK RESULTS ===
Model      | RMSE       | MAE        | MAPE       | MASE       | MSSE      
---------------------------------------------------------------------------
SNAIVE     | 2.2148     | 1.0126     | 11.26%     | 1.5293     | 2.2429
XGB        | 1.6793     | 0.7821     | 9.00%     | 1.1811     | 1.2895
SARIMA     | 1.8523     | 0.8924     | 10.93%     | 1.3477     | 1.5688
TinyTGT    | 1.8341     | 0.9797     | 12.69%     | 1.4795     | 1.5383
---------------------------------------------------------------------------
